In [ ]:
#RBTL24CB061
#!pip -q install spacy scikit-learn
#!python -m spacy download en_core_web_sm
import spacy
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics.pairwise import linear_kernel

nlp = spacy.load("en_core_web_sm")

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [11]:
data = [
    ("Sundar Pichai is the CEO of Google.", "WORKS_FOR"),
    ("Satya Nadella leads Microsoft.", "WORKS_FOR"),
    ("Steve Jobs founded Apple.", "FOUNDED"),
    ("Mark Zuckerberg founded Facebook.", "FOUNDED"),
    ("Google acquired YouTube.", "ACQUIRED"),
    ("Microsoft acquired LinkedIn.", "ACQUIRED"),
    ("Barack Obama was born in Hawaii.", "BORN_IN"),
    ("Elon Musk founded SpaceX.", "FOUNDED")
]

print("Dataset created:", len(data), "sentences")

Dataset created: 8 sentences


In [12]:
def extract_entities_and_path(text):
    doc = nlp(text)

    entities = [
        ent for ent in doc.ents
        if ent.label_ in ["PERSON", "ORG", "GPE", "LOC"]
    ]

    if len(entities) < 2:
        return None

    # Select the first two named entities
    e1, e2 = entities[0], entities[1]

    # Find the shortest dependency path between entity roots
    ancestors1 = [e1.root] + list(e1.root.ancestors)
    ancestors2 = [e2.root] + list(e2.root.ancestors)

    common = next(
        (token for token in ancestors1 if token in ancestors2),
        None
    )

    if common is None:
        path = [e1.root.text, e2.root.text]
    else:
        path1 = ancestors1[:ancestors1.index(common) + 1]
        path2 = ancestors2[:ancestors2.index(common)]

        path = (
            [token.text for token in path1]
            + [token.text for token in reversed(path2)]
        )

    return {
        "entity1": e1.text,
        "entity2": e2.text,
        "entity1_type": e1.label_,
        "entity2_type": e2.label_,
        "path": " -> ".join(path)
    }

for text, label in data:
    print("\nSentence:", text)
    print(extract_entities_and_path(text))


Sentence: Sundar Pichai is the CEO of Google.
{'entity1': 'Sundar Pichai', 'entity2': 'Google', 'entity1_type': 'PERSON', 'entity2_type': 'ORG', 'path': 'Pichai -> is -> CEO -> of -> Google'}

Sentence: Satya Nadella leads Microsoft.
{'entity1': 'Satya Nadella', 'entity2': 'Microsoft', 'entity1_type': 'PERSON', 'entity2_type': 'ORG', 'path': 'Nadella -> leads -> Microsoft'}

Sentence: Steve Jobs founded Apple.
{'entity1': 'Steve Jobs', 'entity2': 'Apple', 'entity1_type': 'PERSON', 'entity2_type': 'ORG', 'path': 'Jobs -> founded -> Apple'}

Sentence: Mark Zuckerberg founded Facebook.
None

Sentence: Google acquired YouTube.
{'entity1': 'Google', 'entity2': 'YouTube', 'entity1_type': 'ORG', 'entity2_type': 'ORG', 'path': 'Google -> acquired -> YouTube'}

Sentence: Microsoft acquired LinkedIn.
{'entity1': 'Microsoft', 'entity2': 'LinkedIn', 'entity1_type': 'ORG', 'entity2_type': 'ORG', 'path': 'Microsoft -> acquired -> LinkedIn'}

Sentence: Barack Obama was born in Hawaii.
{'entity1': 'B

In [13]:
sentences = [text for text, label in data]
labels = [label for text, label in data]

def dependency_features(text):
    result = extract_entities_and_path(text)

    if result is None:
        return "NO_ENTITY_PATH"

    return (
        result["entity1_type"] + " " +
        result["path"].lower() + " " +
        result["entity2_type"]
    )

features = [dependency_features(text) for text in sentences]

# Generate ordered word subsequence features
vectorizer = CountVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    token_pattern=r"(?u)\b\w+\b"
)

X = vectorizer.fit_transform(features)

# Kernel similarity matrix
K = linear_kernel(X, X)

model = SVC(kernel="precomputed")
model.fit(K, labels)

print("Relation extraction model trained successfully!")

Relation extraction model trained successfully!


In [14]:
def predict_relation(text):
    result = extract_entities_and_path(text)

    if result is None:
        print("At least two named entities are required.")
        return

    feature = dependency_features(text)
    X_new = vectorizer.transform([feature])

    # Kernel similarity with training samples
    K_new = linear_kernel(X_new, X)

    prediction = model.predict(K_new)[0]

    print("Sentence:", text)
    print("Entity 1:", result["entity1"])
    print("Entity 2:", result["entity2"])
    print("Dependency Path:", result["path"])
    print("Predicted Relation:", prediction)

predict_relation("Steve Jobs founded Apple.")
predict_relation("Google acquired YouTube.")
predict_relation("Elon Musk founded SpaceX.")

Sentence: Steve Jobs founded Apple.
Entity 1: Steve Jobs
Entity 2: Apple
Dependency Path: Jobs -> founded -> Apple
Predicted Relation: FOUNDED
Sentence: Google acquired YouTube.
Entity 1: Google
Entity 2: YouTube
Dependency Path: Google -> acquired -> YouTube
Predicted Relation: ACQUIRED
At least two named entities are required.


In [15]:
from spacy import displacy

text = "Steve Jobs founded Apple."
doc = nlp(text)

displacy.render(doc, style="dep", jupyter=True)